In [ ]:
# 1. Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder

# 2. Load the Dataset
# We load the primary 'Cleaned_Data' sheet identified in your workbook
file_path = "Meteorological_Dataset_2015-2025.xlsx"
xls = pd.ExcelFile(file_path)
df = pd.read_excel(xls, sheet_name='Cleaned_Data')

print("Initial Dataset Shape:", df.shape)

# 3. Standardize Column Names (Fixing spelling inconsistencies)
df = df.rename(columns={
    'Min. Temperarture': 'Min_Temperature',
    'Max. Temperature': 'Max_Temperature',
    'Wind Dirrection': 'Wind_Direction',
    'Rainefall': 'Rainfall'
})

# Strip any accidental whitespace from column names
df.columns = df.columns.str.strip()

# 4. Handle Duplicates
duplicates_count = df.duplicated().sum()
print(f"Duplicate rows found: {duplicates_count}")
df = df.drop_duplicates()

# 5. Handle Missing Values
# 5a. Target Variable Handling:
# The target variable is 'Evaporation'. If the target is missing, the row is generally useless for supervised training.
initial_rows = len(df)
df = df.dropna(subset=['Evaporation'])
print(f"Dropped {initial_rows - len(df)} rows due to missing target 'Evaporation'.")

# 5b. Predictor Variable Handling:
# For numerical features, we can use median imputation to handle the remaining NaN values (like the 684 missing in 30cm Soil Temp)
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns.drop('Evaporation')
num_imputer = SimpleImputer(strategy='median')
df[numeric_cols] = num_imputer.fit_transform(df[numeric_cols])

# For categorical features (Wind_Direction), we fill missing values with the mode (most frequent)
cat_cols = ['Wind_Direction']
cat_imputer = SimpleImputer(strategy='most_frequent')
df[cat_cols] = cat_imputer.fit_transform(df[cat_cols])

# 6. Categorical Encoding
# Machine learning models require numerical inputs. We use Label Encoding for wind direction.
encoder = LabelEncoder()
df['Wind_Direction_Encoded'] = encoder.fit_transform(df['Wind_Direction'])
df = df.drop(columns=['Wind_Direction']) # Drop the original string column

# 7. Final Sanity Check
print("\n--- Remaining Missing Values ---")
print(df.isnull().sum())
print(f"\nFinal Preprocessed Dataset Shape: {df.shape}")

# 8. Export the Prepared Dataset
# Save the preprocessed dataset so De Silva G.Y.S. can use it for Feature Engineering
output_file = "Preprocessed_Meteorological_Dataset.csv"
df.to_csv(output_file, index=False)
print(f"\nPreprocessed dataset successfully saved to {output_file}")

Initial Dataset Shape: (3378, 24)
Duplicate rows found: 0
Dropped 61 rows due to missing target 'Evaporation'.

--- Remaining Missing Values ---
Date                         0
Dry_Morning                  0
Wet_Morning                  0
Dry_Evening                  0
Wet_Evening                  0
Max_Temperature              0
Min_Temperature              0
S. T. - 5cm Morning          0
S. T. - 10cm Morning         0
S. T. - 20cm Morning         0
S. T. - 30cm Morning         0
S. T. - 5cm Evening          0
S. T. - 10cm Evening         0
S. T. - 20cm Evening         0
S. T. - 30cm Evening         0
R. H. Morning                0
R. H. Evening                0
Mean Velocity                0
SunShine                     0
Rainfall                     0
Evaporation                  0
Rainfall_Trace_Flag          0
Evaporation_Overflow_Flag    0
Wind_Direction_Encoded       0
dtype: int64

Final Preprocessed Dataset Shape: (3317, 24)

Preprocessed dataset successfully saved to Preproce